# 02 - Files and GitHub (Python)

**File:** `notebooks/02_files_and_github_python.ipynb`

**What this does:** Shows how a notebook finds, reads and writes files on disk, and how that work gets back to GitHub.

**How to run it:** Open this file in JupyterLab, check that the kernel shown in the
top-right corner says **Python 3**, then choose *Run > Run All Cells*.

**Inputs:** `data/sample_stations.csv`

**Outputs:** a new file at `outputs/warm_stations.csv`

## Where am I? Finding the repo folder

A notebook runs from the folder it lives in (`notebooks/`), **not** from the top
of the repository. So `data/sample_stations.csv` would not be found, but
`../data/sample_stations.csv` would.

Rather than writing `../` everywhere, the cell below works out where the top of
the repo is once, and builds paths from there. Every notebook here uses this
same short block.

In [ ]:
from pathlib import Path

# Path.cwd() is the folder this notebook runs in.
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

DATA = REPO / "data"

print("Repo folder:", REPO)
print("Data folder:", DATA)

## 1. What files are around me?

`iterdir()` lists the contents of a folder. This is the notebook equivalent of
typing `ls` in a terminal.

In [ ]:
for path in sorted(REPO.iterdir()):
    if path.name.startswith("."):
        continue          # hide .git and friends
    kind = "folder" if path.is_dir() else "file"
    print(f"{kind:7s} {path.name}")

## 2. Reading a file

`sample_stations.csv` is a small example file committed to the repo, so it is
there the moment you clone. `pandas` reads it into a table (a "DataFrame").

In [ ]:
import pandas as pd

stations = pd.read_csv(DATA / "sample_stations.csv")

print(f"{len(stations)} rows, {len(stations.columns)} columns")
stations.head()

## 3. Doing something with it

Keep only the warmer stations. Nothing here changes the file on disk -- `stations`
is a copy held in memory.

In [ ]:
warm = stations[stations["water_temp_c"] > 16]

print(f"{len(warm)} of {len(stations)} stations are warmer than 16 C")
warm[["station_id", "name", "water_temp_c"]]

## 4. Writing a file

Write results to an `outputs/` folder rather than overwriting the input. Getting
into this habit means a mistake never destroys your original data.

In [ ]:
outputs = REPO / "outputs"
outputs.mkdir(exist_ok=True)

destination = outputs / "warm_stations.csv"
warm.to_csv(destination, index=False)

print("Wrote", destination)
print(f"{destination.stat().st_size} bytes")

## 5. Getting your work back to GitHub

A notebook can run shell commands by starting a line with `!`. So you can check
on git without leaving Jupyter:

In [ ]:
!git status --short

The file you just created should appear with a `??` next to it, meaning git can
see it but is not yet tracking it.

To save your work back to GitHub, run these in a terminal
(*File > New > Terminal* in JupyterLab):

```bash
git add .
git commit -m "a short note about what you did"
git push
```

The full walkthrough -- forking, cloning and opening a pull request -- is in the
[main README](../README.md).

Two things worth knowing:

- **Notebooks produce noisy diffs.** A notebook stores its outputs inside the
  file, so re-running it shows up as a change even when you edited nothing.
  Running *Kernel > Restart Kernel and Clear Outputs* before committing keeps
  pull requests readable.
- **Anything in `data/` is deliberately ignored by git** (see `.gitignore`), so
  large downloads never end up in the repository.

## Done

Next: **`03_aquaview_stac_python.ipynb`**, which pulls real data off the internet.